# 멀티모달 미디어 검색 (Gemini Embedding 2, Vector Search 2.0 활용)

이 노트북은 Gemini Embedding 2 모델과 Vector Search 2.0을 활용하여 이미지 및 비디오 등 멀티모달 데이터를 검색하는 시스템을 구축하는 과정을 보여줍니다.

## Phase 1: 환경 설정 및 초기화

프로젝트 ID, 리전, 작업자 수 및 버킷 이름 등 필요한 환경 변수와 상수를 설정합니다. \
⚠️ 꼭 PROJECT_ID 변수 설정을 한 후에 실행해주세요.

In [ ]:
%pip install --upgrade google-cloud-vectorsearch --extra-index-url https://pypi.org/simple
%pip install --upgrade google-cloud-vectorsearch moviepy google-cloud-aiplatform pinecone-text seaborn scikit-learn
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# 테스트 미디어 버킷 이름 지정
import google.auth
_, project_id = google.auth.default()
PROJECT_ID = "your-project-id"
LOCATION = "us-central1"
NUM_WORKER = 32
VIDEO_CHUNK_DURATION = 30   # seconds to chunk video
MAX_IMAGE = 500             # max images to index
MAX_VIDEO = 1               # max videos to index
source_bucket_name = "ai-multimodal-data"

In [ ]:
!gcloud services enable vectorsearch.googleapis.com aiplatform.googleapis.com --project "{PROJECT_ID}"

## Phase 2: 데이터 로드 및 파일 목록 생성

Google Cloud Storage(GCS) 버킷에서 파일 목록을 가져와 이미지와 비디오 파일로 분류합니다.

In [ ]:
# ! 명령어를 통해 gcloud 결과를 file_list 변수에 리스트 형태로 저장
# ** 와일드카드를 사용해 하위 폴더의 모든 파일을 재귀적으로 탐색합니다.
file_list = !gcloud storage ls "gs://{source_bucket_name}/**"

print(f"총 {len(file_list)}개의 파일을 찾았습니다.")
images_list = []
videos_list = []

for file in file_list:
    if file.endswith(('.png', '.jpg')):
        # Convert gs:// to public gcs path
        images_list.append("https://storage.googleapis.com/" + file[5:])
    elif file.endswith('.mp4') and '/vid-chunks/' not in file:
        videos_list.append(file)
print(f"총 {len(images_list)}개의 이미지 파일을 찾았습니다.")
print(f"총 {len(videos_list)}개의 비디오 파일을 찾았습니다.")

### 비디오 청킹(Chunking) 설정

대용량 비디오 파일의 효율적인 임베딩 추출 및 검색을 위해, 비디오를 일정한 시간 단위(예: 30초)로 분할(Chunking)하여 저장할 로컬 디렉토리를 생성합니다.

In [ ]:
# 비디오 Chunking 을 위해 저장할 폴더 생성
import os
# 1. 폴더 생성 (Windows 구문 오류 해결)
os.makedirs("./video/", exist_ok=True)
os.makedirs("./video_chunks/", exist_ok=True)

# for 루프를 통해 비디오를 하나씩 다운로드합니다.
for video in videos_list:
    print(f"다운로드 진행 중: {video}")
    !gcloud storage cp {video} ./video/

### 비디오 파일 필터링

처리할 비디오 파일 목록을 필터링합니다.

In [ ]:
import os
import glob
# 폴더가 섞여 있을 경우를 대비해 파일만 필터링
videos_list = [f for f in glob.glob('./video/*') if os.path.isfile(f)]
print(videos_list)

## Phase 3: 임베딩 추출 유틸리티 및 비디오 처리

비디오 및 이미지에서 임베딩을 추출하기 위한 유틸리티 함수를 정의하고 비디오를 처리합니다.

In [ ]:
# 비디오 및 이미지 임베딩 추출 유틸리티들
from moviepy import VideoFileClip
from google.genai import types
import uuid
import os
def process_chunk_worker(file_path: str, index: int, start_time: float, end_time: float, filename):
    """
    개별 스레드에서 실행될 작업 함수:
    - 비디오의 특정 구간을 자르고 임시 저장
    - 바이트로 변환 후 API 호출
    """
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    # 1. 스레드 독립적인 비디오 객체 생성 (Race condition 방지)
    try:
        video = VideoFileClip(file_path)
        chunk_clip = video.subclipped(start_time, end_time)
    except Exception as e:
        print(f"동영상 로드/자르기 실패 (Chunk {index}): {e}")
        return None

    temp_path = f"./video_chunks/{uuid.uuid4()}.mp4"
    print(f"Processing chunk {index}: {start_time}s - {end_time}s -> {temp_path}")
    
    try:
        # 2. 자른 영상을 임시 파일에 저장 (진행 바 생략을 위해 logger=None)
        chunk_clip.write_videofile(
            temp_path, 
            codec="libx264", 
            audio_codec="aac", 
            logger=None 
        )
        
        # 3. 임시 파일을 읽어서 byte 형태로 변환
        with open(temp_path, "rb") as f:
            chunk_bytes = f.read()
            
        # 4. 임베딩 API 호출
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=[
                types.Part.from_bytes(
                    data=chunk_bytes,
                    mime_type='video/mp4',
                ),
            ],
            config=types.EmbedContentConfig(
                audio_track_extraction=True,
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=5,
                        initial_delay=1.0,
                        max_delay=5.0
                    )
                )
            )
        )
        
        return [start_time, end_time, filename, temp_path, result.embeddings[0].values]
        
    except Exception as e:
        print(f"API 호출 또는 처리 중 오류 발생 (Chunk {index}): {e}")
        return None
        
    finally:
        video.close()

def process_image_worker(file_path: str):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    filename = os.path.basename(file_path)
    
    try:            
        # 임베딩 API 호출
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=[
                types.Part.from_uri(file_uri=file_path),
            ],
            config=types.EmbedContentConfig(
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=5,
                        initial_delay=1.0,
                        max_delay=5.0
                    )
                )
            )
        )
        
        # 비디오와 달리 시간 정보(start, end)가 없으므로 [파일명, 파일경로, 임베딩값] 반환
        return [0, 0, filename, file_path, result.embeddings[0].values]
        
    except Exception as e:
        print(f"API 호출 또는 처리 중 오류 발생 ({filename}): {e}")
        return None

from google import genai
import concurrent.futures
def process_video_parallel(video_path: str, chunk_size: int = 10, max_workers: int = 4):
    """
    비디오를 청크로 나누고, ThreadPoolExecutor를 이용해 병렬로 임베딩을 추출합니다.
    """
    filename = os.path.splitext(os.path.basename(video_path))[0]
    
    # 1. 원본 비디오의 전체 길이 파악
    try:
        with VideoFileClip(video_path) as video:
            duration = video.duration
    except Exception as e:
        print(f"동영상을 불러오는 데 실패했습니다: {e}")
        return []

    # 2. 청크(Chunk) 구간 미리 계산
    chunks = []
    start_time = 0
    index = 0
    while start_time < duration:
        end_time = min(start_time + chunk_size, duration)
        chunks.append((index, start_time, end_time))
        start_time += chunk_size
        index += 1

    results_with_index = []
    
    # 3. 스레드 풀을 이용한 병렬 처리
    print(f"총 {len(chunks)}개의 청크를 {max_workers}개의 스레드로 처리합니다...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 작업을 스레드 풀에 제출 (Future 객체와 청크 인덱스를 매핑)
        future_to_index = {
            executor.submit(process_chunk_worker, video_path, idx, start, end, filename): idx
            for idx, start, end in chunks
        }
        
        # 완료되는 순서대로 결과 수집
        for future in concurrent.futures.as_completed(future_to_index):
            idx = future_to_index[future]
            try:
                res = future.result()
                if res:
                    results_with_index.append((idx, res))
            except Exception as e:
                print(f"스레드 실행 중 예외 발생 (Chunk {idx}): {e}")

    # 4. 병렬 처리로 인해 뒤섞인 결과를 원래 시간순(index)으로 정렬
    results_with_index.sort(key=lambda x: x[0])
    
    # 최종 결과 데이터만 리스트로 반환
    final_results = [res[1] for res in results_with_index]
    return final_results

def process_images_parallel(image_paths: list, max_workers: int = 4):
    """
    여러 장의 이미지를 ThreadPoolExecutor를 이용해 병렬로 임베딩을 추출합니다.
    """
    results_with_index = []
    
    print(f"총 {len(image_paths)}개의 이미지를 {max_workers}개의 스레드로 처리합니다...")
    
    # 스레드 풀을 이용한 병렬 처리
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 입력된 이미지 리스트의 인덱스를 유지하여 순서 보장
        future_to_index = {
            executor.submit(process_image_worker, path): idx
            for idx, path in enumerate(image_paths)
        }
        
        # 완료되는 순서대로 결과 수집
        i = 0
        for future in concurrent.futures.as_completed(future_to_index):
            i += 1
            if i % 100 == 0:
                print(f"{i} 개 처리 완료")
            idx = future_to_index[future]
            try:
                res = future.result()
                if res:
                    results_with_index.append((idx, res))
            except Exception as e:
                print(f"스레드 실행 중 예외 발생 (Image Index {idx}): {e}")

    # 병렬 처리로 인해 뒤섞인 결과를 원래 입력 순서(index)대로 정렬
    results_with_index.sort(key=lambda x: x[0])
    
    # 최종 결과 데이터만 리스트로 반환
    final_results = [res[1] for res in results_with_index]
    return final_results

In [ ]:
# 테스트로 1개 비디오만 선별, Chunking: 30 sec
video_embeddings = process_video_parallel(videos_list[0], chunk_size=VIDEO_CHUNK_DURATION, max_workers=NUM_WORKER)
print(f"{video_embeddings[0][0]}-{video_embeddings[0][1]}-{video_embeddings[0][2]}-{video_embeddings[0][3]}")

### 비디오 임베딩 시각화

t-SNE를 사용하여 비디오 임베딩의 시맨틱 거리를 2차원으로 시각화합니다.

In [ ]:
#비디오 임베딩 정보로 영상의 의미론적 거리 시각화
from sklearn.manifold import TSNE
import numpy as np
import pandas as pd
# 4번째 인덱스에 임베딩 정보를 담고 있음.
video_vectors = [video_embedding[4] for video_embedding in video_embeddings]
time_index = [f"{video_embedding[0]}-{video_embedding[1]}" for video_embedding in video_embeddings]

tsne = TSNE(random_state=0, max_iter=1000)
tsne_results = tsne.fit_transform(
    np.array(video_vectors, dtype=np.float32)
)

df_tsne = pd.DataFrame(tsne_results, columns=["TSNE1", "TSNE2"])
df_tsne["target"] = time_index  # Add labels column from origin data

df_tsne.head()

In [ ]:
#3072차원의 임베딩을 x, y 좌표의 2차원으로 시각화
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(8, 6))  # Set figsize
sns.set_style("darkgrid", {"grid.color": ".6", "grid.linestyle": ":"})
sns.scatterplot(data=df_tsne, x="TSNE1", y="TSNE2", hue="target", palette="hls")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.title("Scatter plot of video using t-SNE")
plt.xlabel("TSNE1")
plt.ylabel("TSNE2")
plt.axis("equal")

### 이미지 처리

이미지 파일에 대한 임베딩을 추출합니다.

In [ ]:
# 테스트로 500개 이미지만 선별
image_embeddings = process_images_parallel(images_list[:MAX_IMAGE], max_workers=NUM_WORKER)
image_embeddings[0]
print(f"{image_embeddings[0][0]}-{image_embeddings[0][1]}-{image_embeddings[0][2]}-{image_embeddings[0][3]}")

## Phase 4: Vector Search 2.0 설정

Vertex AI Vector Search 2.0을 사용하기 위한 서비스 클래스를 초기화하고 데이터 스키마를 정의합니다.

In [ ]:
#Vector Search 2.0 이용을 위한 기본 서비스 클래스 초기화
from google.cloud import vectorsearch_v1beta

vector_search_service_client = vectorsearch_v1beta.VectorSearchServiceClient()
data_object_service_client = vectorsearch_v1beta.DataObjectServiceClient()
data_object_search_service_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

In [ ]:
#데이터 스키마 정의 (비디오, 이미지, 시맨틱 텍스트 모두 content_embedding space를 이용한다고 가정, Gemini embedding 2 이용)
#Vector Search 2.0 에 내장된 string 필드의 Full Text search 정확도가 부족하여 본 예제에서는 Sparse vector 도 활용하여 
#텍스트 Hybrid search도 수행하고자 함. 실습 노트북의 복잡도를 줄이기 위해 사전 학습된 영문 모델(MS MARCO) 이용
from pinecone_text.sparse import BM25Encoder
# 사전 학습된(MS MARCO) 기본 가중치 모델 로드
bm25 = BM25Encoder.default()

COLLECTION_ID = "multimodal-db-001"
collection = vectorsearch_v1beta.Collection(
    data_schema= {
        "type": "object",
        "properties": {            
            "title": {"type": "string"},
            "filename": {"type": "string"},
            "media_type": {"type": "number"},    # 0 for video, 1 for image
            "start": {"type": "number"},
            "end": {"type": "number"},
            "description": {"type": "string"}    # TAGGING for Full Text search using Vector Search 2.0
        },
    },
    vector_schema={
        "content_embedding": {"dense_vector": {"dimensions": 3072}},    # For Video or Image or Semantic Search
        "description_embedding": {"sparse_vector": {}},                 # For Text search using Sparse vector
        #"soundtrack_embedding": {"dense_vector": {"dimensions": 5}},   # Multiple embeddings possible
        #"genre_embedding": {"dense_vector": {"dimensions": 4}},
    },
)
request = vectorsearch_v1beta.CreateCollectionRequest(
    parent=f"projects/{PROJECT_ID}/locations/{LOCATION}",
    collection_id=COLLECTION_ID,
    collection=collection,
)

# Create the collection
operation = vector_search_service_client.create_collection(request=request)
operation.result()

### 데이터 인덱싱

추출된 비디오 및 이미지 임베딩을 Vector Search에 인덱싱합니다.

In [ ]:
#비디오 인덱싱
for embedding in video_embeddings:
    data_object = vectorsearch_v1beta.DataObject(
        data={
            "title": embedding[2],
            "filename": embedding[3],
            "media_type": 0,    # Video
            "start": embedding[0],
            "end": embedding[1]
        },
        vectors={
            "content_embedding": {
                "dense": {"values": embedding[4]}
            }
        },
    )
    request = vectorsearch_v1beta.CreateDataObjectRequest(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
        data_object_id=f"{embedding[2]}-{embedding[0]}-{embedding[1]}",
        data_object=data_object,
    )

    # Make the request
    response = data_object_service_client.create_data_object(request=request)

In [ ]:
#이미지 인덱싱
for embedding in image_embeddings:
    data_object = vectorsearch_v1beta.DataObject(
        data={
            "title": embedding[2],
            "filename": embedding[3],
            "media_type": 1,    # Image
            "start": embedding[0],
            "end": embedding[1]
        },
        vectors={
            "content_embedding": {
                "dense": {"values": embedding[4]}
            }
        },
    )
    
    request = vectorsearch_v1beta.CreateDataObjectRequest(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
        data_object_id=f"{embedding[2]}-{embedding[0]}-{embedding[1]}",
        data_object=data_object,
    )

    # Make the request
    response = data_object_service_client.create_data_object(request=request)

## Phase 5: 미디어 검색 및 렌더링

검색 결과를 시각화하기 위한 렌더링 유틸리티와 검색 함수를 정의합니다.

In [ ]:
# 렌더링 유틸리티
from IPython.display import HTML, display

def display_media_with_titles(media_data, columns=3):
    """
    이미지와 비디오를 타이틀과 함께 지정된 컬럼 수의 타일 형태로 출력합니다.
    media_data: [{'filename': '경로', 'title': '제목'}, ...] 형태의 리스트
    """
    html_code = f"""
    <style>
        .media-grid {{
            display: grid;
            grid-template-columns: repeat({columns}, 1fr);
            gap: 20px;
            margin-top: 10px;
        }}
        .media-card {{
            display: flex;
            flex-direction: column;
            background-color: #f8f9fa;
            padding: 12px;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        }}
        .media-title {{
            font-size: 15px;
            font-weight: 600;
            color: #333;
            margin-bottom: 10px;
            text-align: center;
            word-break: keep-all;
        }}
        /* 비디오와 이미지 모두 동일한 스타일 적용 */
        .media-card video, .media-card img {{
            width: 100%;
            border-radius: 6px;
            object-fit: contain; /* 비율을 유지하면서 꽉 차게 표시 */
            background-color: #000; /* 빈 공간 검은색 처리 (이미지는 필요에 따라 제거 가능) */
        }}
    </style>
    <div class="media-grid">
    """
    
    # 이미지 확장자 목록 정의
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.webp'}
    
    for item in media_data:
        path = item.get('filename', '')
        title = item.get('title', '제목 없음')        
        
        # 파일 확장자 추출 및 소문자 변환
        _, ext = os.path.splitext(path)
        ext = ext.lower()
        
        # 확장자에 따라 태그 분기 처리
        if ext in image_extensions:
            media_tag = f'<img src="{path}" alt="{title}">'
        else:
            media_tag = f"""
            <video controls preload="metadata">
                <source src="{path}">
                브라우저가 비디오 태그를 지원하지 않습니다.
            </video>
            """
            
        html_code += f"""
        <div class="media-card">
            <div class="media-title">{title}</div>
            {media_tag}
        </div>
        """
        
    html_code += "</div>"
    
    display(HTML(html_code))

def render_results(results, columns):
    video_list = []
    for result in results:
        video_list.append({
            'filename': result['filename'],
            'title': f"{result['title']} ({result['end']-result['start']} sec), {result['distance']:0.2f}%"
        })

    # 함수 호출 (6개의 컨텐츠를 3개의 컬럼으로 배치)
    display_media_with_titles(video_list, columns=columns)

In [ ]:
# 벡터 서치 검색
def search(dense_embedding_query, sparse_embedding_query = None, top_k=5, media_type = None, weights = [0.5, 0.5]):
    # media_type 0 은 비디오, 1은 이미지. 필터링에서 특정 조건만 검색하고자 할때 이용
    filter = {}
    if media_type != None:
        filter = {"media_type": {"$eq": media_type}}
    # Hybrid 검색
    if sparse_embedding_query != None:
        request = vectorsearch_v1beta.BatchSearchDataObjectsRequest(
            parent=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
            searches=[
                vectorsearch_v1beta.Search(
                    vector_search=vectorsearch_v1beta.VectorSearch(
                            search_field="content_embedding",
                            vector=vectorsearch_v1beta.DenseVector(values=dense_embedding_query),
                            filter=filter,
                            top_k=top_k,
                            output_fields=vectorsearch_v1beta.OutputFields(
                                data_fields=["*"],
                        ),
                    )
                ),
                vectorsearch_v1beta.Search(
                    vector_search=vectorsearch_v1beta.VectorSearch(
                            search_field="description_embedding",
                            sparse_vector=vectorsearch_v1beta.SparseVector(values=sparse_embedding_query['values'],
                                                                            indices=sparse_embedding_query['indices']),
                            filter=filter,
                            top_k=top_k,
                            output_fields=vectorsearch_v1beta.OutputFields(
                                data_fields=["*"],
                        ),
                    )
                ),

                # Example how to do Full Text search provided by Vector Search 2.0
                #vectorsearch_v1beta.Search(
                #    text_search=vectorsearch_v1beta.TextSearch(
                #            search_text=txt_sparse_query,
                #            data_field_names=["description"],
                #            filter=filter,
                #            top_k=top_k,
                #            output_fields=vectorsearch_v1beta.OutputFields(
                #                data_fields=["*"],
                #        ),
                #    ),
                #)
            ],
            combine=vectorsearch_v1beta.BatchSearchDataObjectsRequest.CombineResultsOptions(
                ranker=vectorsearch_v1beta.Ranker(
                    rrf=vectorsearch_v1beta.ReciprocalRankFusion(weights=weights)
                ),
                top_k=top_k
            ),
        )
        response = data_object_search_service_client.batch_search_data_objects(request=request)
        response = response.results[0].results
    else:
        # Dense vector search
        request = vectorsearch_v1beta.SearchDataObjectsRequest(
            parent=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
            vector_search=vectorsearch_v1beta.VectorSearch(
                    search_field="content_embedding",
                    vector=vectorsearch_v1beta.DenseVector(values=dense_embedding_query),
                    filter=filter,
                    top_k=top_k,
                    output_fields=vectorsearch_v1beta.OutputFields(
                        data_fields=["*"], 
                    ),
                ),
        )
        response = data_object_search_service_client.search_data_objects(request=request)

    results = []
    for item in response:
        # Handle the response
        results.append({
            "distance": item.distance,
            "filename": item.data_object.data["filename"],
            "title": item.data_object.data["title"],
            "start": item.data_object.data["start"],
            "end": item.data_object.data["end"],
        })
    return results

# 텍스트 벡터
def get_text_vector(text):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    result = client.models.embed_content(
        model='gemini-embedding-2',
        contents=[
            types.Part.from_text(text=text),
        ]
    )
    return result.embeddings[0].values

# 텍스트 + 이미지 복합 벡터 (Gemini embedding 2)
def get_text_image_composite_vector(text, image_url):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    result = client.models.embed_content(
        model='gemini-embedding-2',
        contents=[
            types.Part.from_text(text=text),
            types.Part.from_uri(file_uri=image_url)
        ]
    )
    return result.embeddings[0].values

### 텍스트 및 이미지 쿼리 검색

한국어 텍스트 쿼리 및 이미지 쿼리를 사용하여 검색을 수행합니다.

In [ ]:
# Gemini embedding 2를 이용해 Multilingual 로 한국어 쿼리로 이미지, 동영상 검색
results = search(dense_embedding_query=get_text_vector("산속에서 운전"), 
                 top_k = 6)
render_results(results, 3)

In [ ]:
# 리스트에서 MAX_IMAGE 이후 데이터를 테스트로 이용
from IPython.display import Image, display
image_url = images_list[MAX_IMAGE+0]
display(Image(url=image_url))

In [ ]:
# Composite vector 예제 (쿼리벡터: 이미지 + 텍스트, 컨텐츠 소스벡터: 이미지)
# 소스 데이터에 디스크립션이 부족할때 (이미지 + 텍스트로 이미지 검색같은 시나리오, 이미지 + 텍스트로 동영상 검색 등)
results = search(dense_embedding_query=get_text_image_composite_vector("자동차", image_url), 
                 top_k = 6,
                 media_type = 1)
render_results(results, 3)

## Phase 6: 고급 검색 (하이브리드 및 스파스 벡터)

자연어 설명을 추가하여 하이브리드 검색(Dense + Sparse)을 구현합니다.

In [ ]:
# 리스트에서 MAX_IMAGE 이하 이미지(인덱싱된)에 대해 자연어 설명 추가
from IPython.display import Image, display
image_url = images_list[MAX_IMAGE-23]
display(Image(url=image_url))

In [ ]:
# 특정 데이터 오브젝트에 자연어 설명을 추가합니다.
def update_object_data(object_id, description):
    # Initialize request
    data_object = vectorsearch_v1beta.DataObject(
        name=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}/dataObjects/{object_id}",
        data={"description": description},
    )
    request = vectorsearch_v1beta.UpdateDataObjectRequest(
        data_object=data_object,
    )

    # Make the request
    response = data_object_service_client.update_data_object(request=request)

# 특정 데이터 오브젝트에 Sparse vector를 추가합니다.
def update_object_vector(object_id, embedding_vector):
    # Initialize request
    data_object = vectorsearch_v1beta.DataObject(
        name=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}/dataObjects/{object_id}",
        vectors={"description_embedding": {"sparse": embedding_vector}},
    )
    request = vectorsearch_v1beta.UpdateDataObjectRequest(
        data_object=data_object,
    )

    # Make the request
    response = data_object_service_client.update_data_object(request=request)

In [ ]:
# 사진에 자연어 설명(Sparse vector를 이용)을 추가
key = os.path.basename(image_url)
sparse_vector = bm25.encode_documents("Me at the top")
print(sparse_vector)    #단순히 Me, I, at, the 같은 단어들은 불용어 처리된다.
update_object_vector(f"{key}-0-0", sparse_vector)

In [ ]:
sparse_query = "I am on a mountain!"
print(bm25.encode_documents(sparse_query))
results = search(dense_embedding_query=get_text_vector("산"), 
                 sparse_embedding_query=bm25.encode_documents(sparse_query),
                 weights = [0.5, 0.5],
                 top_k = 6,
                 media_type = 1)
render_results(results, 3)

## 리소스 정리

실습 과정에서 생성된 임시 파일 및 데이터 객체 등 사용된 리소스를 정리합니다.

In [ ]:
# 리소스 정리
print("남아있는 데이터 객체를 확인하고 비우는 중...")
collection_path = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
while True:
    # 페이로드 없이 리소스 이름(name)만 조회 (최대 1,000건씩 가져오기)
    query_request = vectorsearch_v1beta.QueryDataObjectsRequest(
        parent=collection_path,
        page_size=1000,
        output_fields=vectorsearch_v1beta.OutputFields(data_fields=[]),
    )
    results = list(data_object_search_service_client.query_data_objects(query_request))
    
    if not results:
        print("비우기 완료")
        break
        
    # 개별 삭제 요청 객체들을 리스트로 구성
    delete_requests = [
        vectorsearch_v1beta.DeleteDataObjectRequest(name=obj.name) 
        for obj in results
    ]
    
    # 일괄 삭제 (Batch Delete) 실행
    data_object_service_client.batch_delete_data_objects(
        parent=collection_path, 
        requests=delete_requests
    )
    print(f"{len(delete_requests)}건 비우기 완료...")

# 3. 빈 Collection 삭제 (Step 2)
print(f"Collection '{COLLECTION_ID}' 삭제 진행 중...")
delete_collection_req = vectorsearch_v1beta.DeleteCollectionRequest(
    name=collection_path
)
operation = vector_search_service_client.delete_collection(request=delete_collection_req)

# 백그라운드 삭제 작업이 완전히 끝날 때까지 대기
operation.result()

print("Collection 삭제 끝")

### 위 예제는 10K 데이터 포인트 이하의 kNN 서버이고, ANN 알고리즘으로 대량의 데이터 기반 검색시 인덱스를 배포하는게 효율적입니다.
### https://docs.cloud.google.com/vertex-ai/docs/vector-search-2/indexes/indexes